# Блокнот 11. Оптимизаторы своими руками

Курс «Машинное обучение с нуля», модуль 11. Разделы: 1–2 — урок 11.2, 3–4 — урок 11.3, 5 — урок 11.4, 6 — урок 11.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Всё на NumPy: сначала оптимизаторы на «долине» $f = \frac{1}{2}(x^2 + 25y^2)$, затем на настоящей сети для рукописных цифр.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

A = np.array([1.0, 25.0])  # крутизна долины по x и по y


def f(p):
    return 0.5 * (A * p ** 2).sum()


def grad(p):
    return A * p


def run(step, lr, steps=500, **kw):
    """Шагает из (−8; 1), пока f не станет меньше 1e−6. Возвращает (число шагов или None, путь)."""
    p = np.array([-8.0, 1.0])
    state, path = {}, [p.copy()]
    for t in range(1, steps + 1):
        p = step(p, grad(p), state, lr, t, **kw)
        path.append(p.copy())
        if f(p) < 1e-6:
            return t, np.array(path)
        if not np.isfinite(f(p)) or f(p) > 1e6:
            break
    return None, np.array(path)


def show_paths(runs, title):
    xx, yy = np.meshgrid(np.linspace(-9, 4, 300), np.linspace(-1.6, 1.6, 200))
    plt.figure(figsize=(9, 3))
    plt.contour(xx, yy, 0.5 * (xx ** 2 + 25 * yy ** 2), levels=np.geomspace(0.05, 45, 12), colors="gray", linewidths=0.5)
    for name, path in runs.items():
        plt.plot(*path[:60].T, "-o", ms=2, label=name)
    plt.scatter(0, 0, marker="*", s=150, color="k", zorder=5)
    plt.xlim(-9, 4)
    plt.ylim(-1.6, 1.6)
    plt.legend()
    plt.title(title)
    plt.show()

## 1. Простой градиентный спуск на долине (урок 11.2)

Шаг $w \leftarrow w - \eta g$. Предел устойчивости — $2/25 = 0{,}08$.

In [ ]:
def sgd(p, g, s, lr, t):
    return p - lr * g


for lr in (0.01, 0.03, 0.05, 0.079, 0.085):
    print(f"SGD, шаг {lr}: шагов до цели {run(sgd, lr)[0]}")
show_paths({"шаг 0,03": run(sgd, 0.03)[1], "шаг 0,07": run(sgd, 0.07)[1]}, "Простой SGD: медленно или зигзагом")

## 2. Момент и Нестеров (урок 11.2)

**Задание 1.** Допишите шаг SGD с моментом в записи PyTorch: `v = beta * v + g`, затем `w - lr * v`; для Нестерова — `w - lr * (g + beta * v)`. Скорость `v` хранится в словаре состояния `s` (в начале её там нет — берите нули).

In [ ]:
def momentum(p, g, s, lr, t, beta=0.9, nesterov=False):
    v = s.get("v", np.zeros_like(p))
    ...  # ваш код: обновите v, сохраните в s["v"], верните новые веса

In [ ]:
for lr in (0.01, 0.03):
    print(f"шаг {lr}: момент — {run(momentum, lr)[0]} шагов, Нестеров — {run(momentum, lr, nesterov=True)[0]} шагов")
assert run(momentum, 0.03)[0] == 141 and run(momentum, 0.03, nesterov=True)[0] == 67
show_paths({"момент 0,9": run(momentum, 0.03)[1], "Нестеров": run(momentum, 0.03, nesterov=True)[1]}, "Шаг 0,03")

Попробуйте другой коэффициент момента: для этой долины теория даёт наилучшее $\beta = \left(\frac{\sqrt{25} - 1}{\sqrt{25} + 1}\right)^2 \approx 0{,}44$ при подходящем шаге. Найдите сочетание шага и $\beta$, при котором момент доходит до цели меньше чем за 60 шагов.

In [ ]:
best = min((run(momentum, lr, beta=b)[0] or 10 ** 9, lr, b) for lr in np.linspace(0.01, 0.15, 29) for b in (0.0, 0.3, 0.44, 0.6, 0.9))
print("лучшее: шагов {}, шаг {:.3f}, beta {}".format(*best))

## 3. Adam своими руками (урок 11.3)

**Задание 2.** Допишите шаг Adam с поправкой смещения: скользящие средние `m` и `v`, затем `m_hat`, `v_hat` и шаг.

In [ ]:
def adam(p, g, s, lr, t, b1=0.9, b2=0.999, eps=1e-8):
    m, v = s.get("m", np.zeros_like(p)), s.get("v", np.zeros_like(p))
    ...  # ваш код

In [ ]:
# Первый шаг Adam сдвигает каждую координату ровно на lr против знака градиента
p1 = adam(np.array([2.0, 2.0]), np.array([-0.04, 300.0]), {}, 0.01, 1)
print("после первого шага:", p1)
assert np.allclose(p1, [2.01, 1.99])
for lr in (0.05, 0.1, 0.3, 1.0):
    print(f"Adam, шаг {lr}: шагов до цели {run(adam, lr, steps=2000)[0]}")
show_paths({"Adam 0,3": run(adam, 0.3)[1], "Нестеров 0,03": run(momentum, 0.03, nesterov=True)[1]}, "Adam и Нестеров")

## 4. Оптимизаторы на настоящей сети (урок 11.3)

Сеть 64 → 64 → 10 с ReLU и softmax на рукописных цифрах (8×8 пикселей), батч 64, 20 эпох. Прямой и обратный проходы — как в блокноте модуля 10. Оптимизаторы — ваши `sgd`, `momentum`, `adam`: каждый вес обновляется своей функцией со своим состоянием.

In [ ]:
X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)


def init_net(seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, np.sqrt(2 / 64), (64, 64)), "b1": np.zeros(64),
            "W2": r.normal(0, np.sqrt(2 / 64), (64, 10)), "b2": np.zeros(10)}


def forward_backward(net, Xb, yb):
    Z1 = Xb @ net["W1"] + net["b1"]
    H = np.maximum(0, Z1)
    Z2 = H @ net["W2"] + net["b2"]
    P = np.exp(Z2 - Z2.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    n = len(yb)
    loss = -np.log(P[np.arange(n), yb] + 1e-12).mean()
    dZ2 = P.copy()
    dZ2[np.arange(n), yb] -= 1
    dZ2 /= n                                   # градиент softmax + CE: (p − y) / n
    dH = dZ2 @ net["W2"].T
    dZ1 = dH * (Z1 > 0)
    return loss, {"W1": Xb.T @ dZ1, "b1": dZ1.sum(0), "W2": H.T @ dZ2, "b2": dZ2.sum(0)}


def accuracy(net, X, y):
    H = np.maximum(0, X @ net["W1"] + net["b1"])
    return np.mean((H @ net["W2"] + net["b2"]).argmax(1) == y)


def train(step, lr, epochs=20, batch=64, schedule=None, clip=None, seed=0, **kw):
    net, states, t = init_net(seed), {k: {} for k in ("W1", "b1", "W2", "b2")}, 0
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        cur_lr = schedule(epoch) if schedule else lr
        idx = rng.permutation(len(X_tr))
        losses = []
        for start in range(0, len(idx), batch):
            b = idx[start:start + batch]
            loss, grads = forward_backward(net, X_tr[b], y_tr[b])
            if clip is not None:
                total = np.sqrt(sum((g ** 2).sum() for g in grads.values()))
                grads = {k: g * min(1.0, clip / total) for k, g in grads.items()}
            t += 1
            for k in net:
                net[k] = step(net[k], grads[k], states[k], cur_lr, t, **kw)
            losses.append(loss)
        history.append(np.mean(losses))
    return net, history

In [ ]:
results = {}
for name, step, lr in (("SGD 0,01", sgd, 0.01), ("момент 0,01", momentum, 0.01), ("Нестеров 0,01", lambda *a: momentum(*a, nesterov=True), 0.01),
                       ("Adam 0,001", adam, 0.001)):
    net, hist = train(step, lr)
    results[name] = hist
    print(f"{name}: потеря {hist[-1]:.3f}, тест {accuracy(net, X_te, y_te):.3f}")
    plt.plot(range(1, 21), hist, label=name)
plt.yscale("log")
plt.xlabel("эпоха")
plt.ylabel("потеря на обучении")
plt.legend()
plt.show()
assert results["момент 0,01"][-1] < results["SGD 0,01"][-1]

## 5. Расписание скорости обучения (урок 11.4)

**Задание 3.** Допишите прогрев + косинус: при `t < warm` — `lr_max * (t + 1) / warm`, иначе `0.5 * lr_max * (1 + cos(pi * (t - warm) / (T - warm)))`.

In [ ]:
def cosine_warmup_lr(t, T, lr_max, warm):
    ...  # ваш код

In [ ]:
lrs = [cosine_warmup_lr(t, 100, 0.1, 10) for t in range(100)]
assert np.allclose([lrs[0], lrs[4], lrs[9], lrs[55]], [0.01, 0.05, 0.1, 0.05])
plt.plot(lrs)
plt.xlabel("эпоха")
plt.ylabel("скорость обучения")
plt.title("Прогрев 10 эпох + косинус")
plt.show()

Сравним на сети постоянный шаг 0,1 и то же начало с прогревом 2 эпохи и косинусом на 20 эпохах (SGD с моментом).

In [ ]:
for name, sched in (("постоянный 0,1", None), ("прогрев + косинус", lambda e: cosine_warmup_lr(e, 20, 0.1, 2))):
    net, hist = train(momentum, 0.1, schedule=sched)
    print(f"{name}: потеря {hist[-1]:.4f}, тест {accuracy(net, X_te, y_te):.3f}")

На маленькой сети и 20 эпохах разница невелика — честный результат. Расписания сильнее всего помогают при долгом обучении больших сетей, где без спада шага шум батчей не даёт опуститься на дно.

## 6. Обрезка градиента и числа fp16 (урок 11.5)

**Задание 4.** Допишите обрезку по норме: если длина `g` больше `max_norm`, умножьте `g` на `max_norm / ‖g‖`.

In [ ]:
def clip_by_norm(g, max_norm):
    ...  # ваш код

In [ ]:
assert np.allclose(clip_by_norm(np.array([3.0, 4.0]), 1), [0.6, 0.8])
assert np.allclose(clip_by_norm(np.array([0.3, 0.4]), 1), [0.3, 0.4])
print("Обрезка работает!")

# Слишком большой шаг без обрезки и с ней
for clip in (None, 1.0):
    net, hist = train(momentum, 0.5, epochs=10, clip=clip)
    print(f"шаг 0,5, обрезка {clip}: потеря {hist[-1]:.3f}, тест {accuracy(net, X_te, y_te):.3f}")

С шагом 0,5 без обрезки отдельные огромные градиенты выбрасывают веса, и сеть учится плохо; обрезка по норме 1 сохраняет обучение.

Наконец, как ведут себя числа в fp16 — формате смешанной точности.

In [ ]:
print("наибольшее fp16:", np.finfo(np.float16).max)
print("70 000 в fp16:", np.float16(70000))
print("1e-8 в fp16:", np.float16(1e-8))
print("1e-8 · 1024 в fp16:", np.float16(1e-8 * 1024), "→ обратно в fp32:", np.float32(np.float16(1e-8 * 1024)) / 1024)
print("1 + 0,0001 в fp16:", np.float16(1) + np.float16(0.0001))
assert np.float16(1e-8) == 0 and np.isinf(np.float16(70000))

Последняя строка показывает ещё одну причину держать главную копию весов в fp32: маленький шаг $10^{-4}$, прибавленный к весу 1 в fp16, теряется целиком — точности не хватает.